## Baseline estimation — rounds 1 and 2


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
# BigQuery'den paneli çek
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT 
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# Güvenli tipler
df["chosen"] = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)
if "obs_weight" not in df.columns or df["obs_weight"].isna().all():
    df["obs_weight"] = 1.0

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
df.head(3)
# Tasarım matrisini kur: X = [ListenScore, TimeCost, Sponsor]
X_cols = ["listen_score_z", "timecost_z", "sponsor"]
X = df[X_cols].to_numpy(dtype=float)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

# Her kullanıcı seti bir "choice situation": y, X bu set içinde 1 seçilmiş + diğerleri 0
# Gruplama indeksleri:
uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
K = X.shape[1]
N = uniq_users.size

# Her kullanıcı için başlangıç-bitiş indeksleri:
order = np.argsort(users)
# Ama dataframe zaten kullanıcıya göre sıralıysa gerek yok; yine de savunmacı:
df = df.iloc[order].reset_index(drop=True)
X = df[X_cols].to_numpy()
y = df["chosen"].to_numpy()
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy()
uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)

# Kullanıcı ağırlığını user-bazında sabitle (satır bazında da eşit ise zaten aynı)
# Burada her kullanıcının w_i = ortalama obs_weight'i:
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)])

def logsumexp(a):
    amax = np.max(a)
    return amax + np.log(np.sum(np.exp(a - amax)))

def mnl_obj(beta):
    """ Negatif log-likelihood (ağırlıklı). """
    beta = np.asarray(beta)
    ll = 0.0
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        # log-likelihood katkısı: x_chosen'beta - logsumexp(Xbeta)
        ll_i = (ui[yi==1].sum()) - logsumexp(ui)
        ll += w_user[idx_u] * ll_i
    return -ll

def mnl_score(beta):
    """ Gradient (ilk türev), ağırlıklı; per-user score'ların toplamı. """
    beta = np.asarray(beta)
    grad = np.zeros(K)
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        # softmax olasılıkları
        expu = np.exp(ui - np.max(ui))
        p = expu / expu.sum()
        # score_i = x_chosen - sum_j p_j x_j
        x_chosen = Xi[yi==1].sum(axis=0)  # (tek 1 varsayıyoruz)
        x_bar = Xi.T @ p
        grad += w_user[idx_u] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    """ Hessian (ikinci türev), beklenen bilgi; negatif definite (maksimizasyonda). """
    beta = np.asarray(beta)
    H = np.zeros((K, K))
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        expu = np.exp(ui - np.max(ui))
        p = expu / expu.sum()
        # Var_p[X] = sum_j p_j (x_j x_j') - (sum_j p_j x_j)(sum_j p_j x_j')'
        X_weighted = Xi * p[:, None]
        x_bar = X_weighted.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[idx_u] * S
    return H

# Başlangıç (0 vektörü genelde çalışır; istersen küçük değerler ver)
beta0 = np.zeros(K)

opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x
H = mnl_hessian(beta_hat)  # bread
# Kullanıcı-cluster "meat": sum_i (score_i score_i')
meat = np.zeros((K, K))
for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]
    yi = y[s:s+c]
    ui = Xi @ beta_hat
    expu = np.exp(ui - np.max(ui))
    p = expu / expu.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[idx_u] * (x_chosen - x_bar)   # (K,)
    meat += np.outer(score_i, score_i)

# Sandwich cov = inv(H) * meat * inv(H)'
H_inv = np.linalg.inv(H)
cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# LL ve McFadden R^2
LL = -mnl_obj(beta_hat)
# Null model: sadece sabit; alternatif-varying sabit MNL'de tanımsız; 
# burada "sabit yok" null LL ≈ her set için log(alt sayısı) kabul edilebilir ~ sum(-log(J_i))
LL0 = 0.0
for s, c in zip(start_idx, counts):
    LL0 += -np.log(c)
mcfadden_r2 = 1 - (LL / LL0)

# Sonuç tablosu
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": beta_hat / se
}, index=["ListenScore", "TimeCost", "Sponsor"])

print("=== MNL (etkileşimsiz) — Sonuçlar ===")
print(res.round(4))
print(f"\nLog-Likelihood: {LL:.3f}")
print(f"McFadden R^2 (approx): {mcfadden_r2:.4f}")
# V_ij = b1*L - b2*T + b3*Sponsor
b1, b2, b3 = beta_hat
df_sorted = df.sort_values(["user_id", "podcast_id"]).copy()
V = b1*df_sorted["listen_score_z"].to_numpy() - b2*df_sorted["timecost_z"].to_numpy() + b3*df_sorted["sponsor"].to_numpy()

# CS_i = log sum_j exp(V_ij)
cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(np.log(np.exp(Vi - Vi.max()).sum()) + Vi.max())
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS")

print("\n=== Consumer Surplus (per user) ===")
print(cs.describe())

# Counterfactual: Sponsor=0
V_cf = b1*df_sorted["listen_score_z"].to_numpy() - b2*df_sorted["timecost_z"].to_numpy() + b3*0
cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(np.log(np.exp(Vi - Vi.max()).sum()) + Vi.max())
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_no_sponsor")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) ===")
print(delta.describe())


## Baseline estimation — round 3


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
# BigQuery'den paneli çek
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# Güvenli tipler
df["chosen"] = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)
if "obs_weight" not in df.columns or df["obs_weight"].isna().all():
    df["obs_weight"] = 1.0

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
df.head(3)
# Tasarım matrisini kur: X = [ListenScore, TimeCost, Sponsor]
X_cols = ["listen_score_z", "timecost_z", "sponsor"]
X = df[X_cols].to_numpy(dtype=float)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

# Her kullanıcı seti bir "choice situation": y, X bu set içinde 1 seçilmiş + diğerleri 0
# Gruplama indeksleri:
uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
K = X.shape[1]
N = uniq_users.size

# Her kullanıcı için başlangıç-bitiş indeksleri:
order = np.argsort(users)
# Ama dataframe zaten kullanıcıya göre sıralıysa gerek yok; yine de savunmacı:
df = df.iloc[order].reset_index(drop=True)
X = df[X_cols].to_numpy()
y = df["chosen"].to_numpy()
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy()
uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)

# Kullanıcı ağırlığını user-bazında sabitle (satır bazında da eşit ise zaten aynı)
# Burada her kullanıcının w_i = ortalama obs_weight'i:
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)])

def logsumexp(a):
    amax = np.max(a)
    return amax + np.log(np.sum(np.exp(a - amax)))

def mnl_obj(beta):
    """ Negatif log-likelihood (ağırlıklı). """
    beta = np.asarray(beta)
    ll = 0.0
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        # log-likelihood katkısı: x_chosen'beta - logsumexp(Xbeta)
        ll_i = (ui[yi==1].sum()) - logsumexp(ui)
        ll += w_user[idx_u] * ll_i
    return -ll

def mnl_score(beta):
    """ Gradient (ilk türev), ağırlıklı; per-user score'ların toplamı. """
    beta = np.asarray(beta)
    grad = np.zeros(K)
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        # softmax olasılıkları
        expu = np.exp(ui - np.max(ui))
        p = expu / expu.sum()
        # score_i = x_chosen - sum_j p_j x_j
        x_chosen = Xi[yi==1].sum(axis=0)  # (tek 1 varsayıyoruz)
        x_bar = Xi.T @ p
        grad += w_user[idx_u] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    """ Hessian (ikinci türev), beklenen bilgi; negatif definite (maksimizasyonda). """
    beta = np.asarray(beta)
    H = np.zeros((K, K))
    for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        expu = np.exp(ui - np.max(ui))
        p = expu / expu.sum()
        # Var_p[X] = sum_j p_j (x_j x_j') - (sum_j p_j x_j)(sum_j p_j x_j')'
        X_weighted = Xi * p[:, None]
        x_bar = X_weighted.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[idx_u] * S
    return H

# Başlangıç (0 vektörü genelde çalışır; istersen küçük değerler ver)
beta0 = np.zeros(K)

opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x
H = mnl_hessian(beta_hat)  # bread
# Kullanıcı-cluster "meat": sum_i (score_i score_i')
meat = np.zeros((K, K))
for idx_u, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]
    yi = y[s:s+c]
    ui = Xi @ beta_hat
    expu = np.exp(ui - np.max(ui))
    p = expu / expu.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[idx_u] * (x_chosen - x_bar)   # (K,)
    meat += np.outer(score_i, score_i)

# Sandwich cov = inv(H) * meat * inv(H)'
H_inv = np.linalg.inv(H)
cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

LL = -mnl_obj(beta_hat)

# LL0: eşit olasılık null modeli, KULLANICI AĞIRLIĞIYLA
# J_i = o kullanıcı setindeki alternatif sayısı (counts)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))

mcfadden_r2 = 1 - (LL / LL0)

# (Opsiyonel) McFadden Adjusted R^2
# k = parametre sayısı, N = kullanıcı sayısı (choice situations)
k = X.shape[1]
N_users = len(counts)
mcfadden_adj = 1 - ((LL - k) / LL0)

print(f"\nLog-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# Sonuç tablosu
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": beta_hat / se
}, index=["ListenScore", "TimeCost", "Sponsor"])

print("=== MNL (etkileşimsiz) — Sonuçlar ===")
print(res.round(4))
print(f"\nLog-Likelihood: {LL:.3f}")
print(f"McFadden R^2 (approx): {mcfadden_r2:.4f}")
# V_ij = b1*L - b2*T + b3*Sponsor
b1, b2, b3 = beta_hat
df_sorted = df.sort_values(["user_id", "podcast_id"]).copy()
V = b1*df_sorted["listen_score_z"].to_numpy() - b2*df_sorted["timecost_z"].to_numpy() + b3*df_sorted["sponsor"].to_numpy()

# CS_i = log sum_j exp(V_ij)
cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(np.log(np.exp(Vi - Vi.max()).sum()) + Vi.max())
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS")

print("\n=== Consumer Surplus (per user) ===")
print(cs.describe())

# Counterfactual: Sponsor=0
V_cf = b1*df_sorted["listen_score_z"].to_numpy() - b2*df_sorted["timecost_z"].to_numpy() + b3*0
cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(np.log(np.exp(Vi - Vi.max()).sum()) + Vi.max())
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_no_sponsor")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) ===")
print(delta.describe())


## Model 2 version 1


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# ---------- 0) BigQuery'den M2 verisini çek ----------
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  listen_highEdu,
  time_female,
  sponsor_young,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# ---------- 1) Tipleri sabitle ----------
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)

num_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_highEdu", "time_female", "sponsor_young",
    "obs_weight"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---------- 2) Tasarım matrisi (M1 + 3 interaction) ----------
X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_highEdu", "time_female", "sponsor_young"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "ListenScore×HighEdu", "TimeCost×Female", "Sponsor×Young"
]

# Sıralama ve grup indeksleri (user bazlı choice situations)
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)

# user-level weight: her kullanıcının ortalama satır ağırlığı
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)

K = X.shape[1]

# ---------- 3) MNL: LL, gradient, hessian ----------
def mnl_obj(beta):
    """Negatif log-likelihood (user-weighted)."""
    beta = np.asarray(beta, dtype=np.float64)
    ll = 0.0
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        ll_i = ui[yi==1].sum() - logsumexp(ui)
        ll += w_user[i] * ll_i
    return -ll

def mnl_score(beta):
    """Gradient (user-weighted)."""
    beta = np.asarray(beta, dtype=np.float64)
    grad = np.zeros(K, dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        x_chosen = Xi[yi==1].sum(axis=0)  # tek 1 varsayımı
        x_bar = Xi.T @ p
        grad += w_user[i] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    """Beklenen bilgi matrisi (negatif definite)."""
    beta = np.asarray(beta, dtype=np.float64)
    H = np.zeros((K, K), dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        Xw = Xi * p[:, None]
        x_bar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[i] * S
    return H

# ---------- 4) MLE ----------
beta0 = np.zeros(K, dtype=np.float64)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# Cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat)
H_inv = np.linalg.inv(H)
meat = np.zeros((K, K), dtype=np.float64)
for i, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]
    yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p = p / p.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[i] * (x_chosen - x_bar)
    meat += np.outer(score_i, score_i)

cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---------- 5) LL, LL0 (weighted), R^2 ----------
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
mcfadden_r2 = 1 - (LL / LL0)
k = K
N_users = len(counts)
mcfadden_adj = 1 - ((LL - k) / LL0)

# ---------- 6) Sonuç tablosu ----------
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== MNL (Model 2: Demografi × Özellik) — Sonuçlar ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# ---------- 7) CS ve karşı-olgusal (Sponsor=0) ----------
# Not: utility = b1*L - b2*T + b3*S + gamma*interactions
b = dict(zip(param_names, beta_hat))

# V = baseline + interactions
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
LxEdu = df["listen_highEdu"].to_numpy()
TxFem = df["time_female"].to_numpy()
SxYoung = df["sponsor_young"].to_numpy()

V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["ListenScore×HighEdu"]*LxEdu
     + b["TimeCost×Female"]*TxFem
     + b["Sponsor×Young"]*SxYoung)

# CS_i = log sum_j exp(V_ij)
cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(logsumexp(Vi))
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS_M2")
print("\n=== Consumer Surplus (per user) — M2 ===")
print(cs.describe())

# Counterfactual: Sponsor=0 (interactions da 0 olur çünkü S=0 → SxYoung=0)
V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + b["ListenScore×HighEdu"]*LxEdu
        + b["TimeCost×Female"]*TxFem
        + 0.0)

cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(logsumexp(Vi))
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_noSponsor_M2")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) — M2 ===")
print(delta.describe())


## Model 2 version 2


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# ---------- 0) BigQuery’den M2V2 verisini çek ----------
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V2"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  sponsor_highIncome,
  time_highFreq,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# ---------- 1) Tipleri sabitle (M2V1 ile aynı) ----------
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)

num_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "sponsor_highIncome", "time_highFreq",
    "obs_weight"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---------- 2) Tasarım matrisi (M2V2 farkı: yeni 2 interaction) ----------
X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "sponsor_highIncome", "time_highFreq"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "Sponsor×HighIncome", "TimeCost×HighFreq"
]

# Sıralama ve grup indeksleri (user bazlı choice situations)
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)

K = X.shape[1]

# ---------- 3) MNL: LL, gradient, hessian (M2V1 ile aynı) ----------
def mnl_obj(beta):
    beta = np.asarray(beta, dtype=np.float64)
    ll = 0.0
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        ll += w_user[i] * (ui[yi==1].sum() - logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, dtype=np.float64)
    grad = np.zeros(K, dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        x_chosen = Xi[yi==1].sum(axis=0)
        x_bar = Xi.T @ p
        grad += w_user[i] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    beta = np.asarray(beta, dtype=np.float64)
    H = np.zeros((K, K), dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        Xw = Xi * p[:, None]
        x_bar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[i] * S
    return H

# ---------- 4) MLE ----------
beta0 = np.zeros(K, dtype=np.float64)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# Cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat)
H_inv = np.linalg.inv(H)
meat = np.zeros((K, K), dtype=np.float64)
for i, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]; yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p = p / p.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[i] * (x_chosen - x_bar)
    meat += np.outer(score_i, score_i)

cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---------- 5) LL, LL0 (weighted), R² ----------
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
mcfadden_r2 = 1 - (LL / LL0)
mcfadden_adj = 1 - ((LL - K) / LL0)

# ---------- 6) Sonuç tablosu ----------
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== MNL (Model 2 — V2) Sonuçlar ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# ---------- 7) CS ve karşı-olgusal (Sponsor=0) ----------
b = dict(zip(param_names, beta_hat))

L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
SxInc = df["sponsor_highIncome"].to_numpy()
TxFreq = df["time_highFreq"].to_numpy()

V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["Sponsor×HighIncome"]*SxInc
     + b["TimeCost×HighFreq"]*TxFreq)

# CS_i = log sum_j exp(V_ij)
cs_list, ptr = [], 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(logsumexp(Vi))
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS_M2V2")
print("\n=== Consumer Surplus (per user) — M2V2 ===")
print(cs.describe())

# Counterfactual: Sponsor=0 → S=0 ve SxInc=0
V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + 0.0
        + b["TimeCost×HighFreq"]*TxFreq)

cs_cf_list, ptr = [], 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(logsumexp(Vi))
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_noSponsor_M2V2")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) — M2V2 ===")
print(delta.describe())


## Model 2 version 3.1


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# ---------- 0) BigQuery'den M2V3 verisini çek ----------
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V3"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  listen_young,
  sponsor_prefComedy,
  sponsor_prefPolitics,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# ---------- 1) Tipleri sabitle ----------
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)

num_cols = [
    "listen_score_z","timecost_z","sponsor",
    "listen_young","sponsor_prefComedy","sponsor_prefPolitics",
    "obs_weight"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---------- 2) Tasarım matrisi ----------
X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_young", "sponsor_prefComedy", "sponsor_prefPolitics"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "ListenScore×Young", "Sponsor×PrefComedy", "Sponsor×PrefPolitics"
]

# Sıralama ve grup indeksleri (user bazlı choice situations)
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)

# user-level weight: her kullanıcının ortalama satır ağırlığı
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)

K = X.shape[1]

# ---------- 3) MNL: LL, gradient, hessian ----------
def mnl_obj(beta):
    beta = np.asarray(beta, dtype=np.float64)
    ll = 0.0
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        ll_i = ui[yi==1].sum() - logsumexp(ui)
        ll += w_user[i] * ll_i
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, dtype=np.float64)
    grad = np.zeros(K, dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        x_chosen = Xi[yi==1].sum(axis=0)
        x_bar = Xi.T @ p
        grad += w_user[i] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    beta = np.asarray(beta, dtype=np.float64)
    H = np.zeros((K, K), dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        Xw = Xi * p[:, None]
        x_bar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[i] * S
    return H

# ---------- 4) MLE ----------
beta0 = np.zeros(K, dtype=np.float64)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# Cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat)
H_inv = np.linalg.inv(H)
meat = np.zeros((K, K), dtype=np.float64)
for i, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]
    yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p = p / p.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[i] * (x_chosen - x_bar)
    meat += np.outer(score_i, score_i)

cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---------- 5) LL, LL0 (weighted), R^2 ----------
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
mcfadden_r2 = 1 - (LL / LL0)
k = K
N_users = len(counts)
mcfadden_adj = 1 - ((LL - k) / LL0)

# ---------- 6) Sonuç tablosu ----------
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== MNL (Model 2 V3) — Sonuçlar ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# ---------- 7) CS ve karşı-olgusal (Sponsor=0) ----------
b = dict(zip(param_names, beta_hat))

L  = df["listen_score_z"].to_numpy()
T  = df["timecost_z"].to_numpy()
S  = df["sponsor"].to_numpy()
Ly = df["listen_young"].to_numpy()
Sc = df["sponsor_prefComedy"].to_numpy()
Sp = df["sponsor_prefPolitics"].to_numpy()

# V (gerçek durum)
V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["ListenScore×Young"]*Ly
     + b["Sponsor×PrefComedy"]*Sc
     + b["Sponsor×PrefPolitics"]*Sp)

# CS_i = log sum_j exp(V_ij)
cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(logsumexp(Vi))
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS_M2V3")
print("\n=== Consumer Surplus (per user) — M2V3 ===")
print(cs.describe())

# Counterfactual: Sponsor = 0 (sponsor etkileşimleri de 0'a düşer)
V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + b["ListenScore×Young"]*Ly
        + 0.0
        + 0.0)

cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(logsumexp(Vi))
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_noSponsor_M2V3")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) — M2V3 ===")
print(delta.describe())


## Model 2 version 3.2


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# ---------- 0) BigQuery'den M2V3 verisini çek ----------
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V3"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  listen_young,
  time_highIncome,
  listen_highFreq,
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# ---------- 1) Tipleri sabitle ----------
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)

num_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_young", "time_highIncome", "listen_highFreq",
    "obs_weight"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---------- 2) X matrisi (baseline + M2V3 interactions) ----------
X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_young", "time_highIncome", "listen_highFreq"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "ListenScore×Young", "TimeCost×HighIncome", "ListenScore×HighFreq"
]

# Sıralama ve grup indeksleri (user bazlı)
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)

K = X.shape[1]

# ---------- 3) MNL fonksiyonları ----------
def mnl_obj(beta):
    beta = np.asarray(beta, dtype=np.float64)
    ll = 0.0
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        ll_i = ui[yi==1].sum() - logsumexp(ui)
        ll  += w_user[i] * ll_i
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, dtype=np.float64)
    grad = np.zeros(K, dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        x_chosen = Xi[yi==1].sum(axis=0)
        x_bar = Xi.T @ p
        grad += w_user[i] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    beta = np.asarray(beta, dtype=np.float64)
    H = np.zeros((K, K), dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        Xw = Xi * p[:, None]
        x_bar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[i] * S
    return H

# ---------- 4) MLE ----------
beta0 = np.zeros(K, dtype=np.float64)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# Cluster-robust SE (user)
H = mnl_hessian(beta_hat)
H_inv = np.linalg.inv(H)
meat = np.zeros((K, K), dtype=np.float64)
for i, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]; yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p /= p.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[i] * (x_chosen - x_bar)
    meat += np.outer(score_i, score_i)

cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---------- 5) LL, LL0 (weighted), R^2 ----------
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
mcfadden_r2 = 1 - (LL / LL0)
mcfadden_adj = 1 - ((LL - K) / LL0)

# ---------- 6) Sonuçlar ----------
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== MNL (Model 2 — V3) Sonuçlar ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# ---------- 7) CS ve karşı-olgusal (Sponsor=0) ----------
b = dict(zip(param_names, beta_hat))
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
Ly = df["listen_young"].to_numpy()
Th = df["time_highIncome"].to_numpy()
Lh = df["listen_highFreq"].to_numpy()

V = (b["ListenScore"]*L + b["TimeCost"]*T + b["Sponsor"]*S
     + b["ListenScore×Young"]*Ly
     + b["TimeCost×HighIncome"]*Th
     + b["ListenScore×HighFreq"]*Lh)

cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(logsumexp(Vi))
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS_M2V3")
print("\n=== Consumer Surplus (per user) — M2V3 ===")
print(cs.describe())

# Sponsor=0 karşı-olgusal (sponsor içeren terimler 0'a düşer, diğer interaction'lar kalır)
V_cf = (b["ListenScore"]*L + b["TimeCost"]*T + 0.0
        + b["ListenScore×Young"]*Ly
        + b["TimeCost×HighIncome"]*Th
        + b["ListenScore×HighFreq"]*Lh)

cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(logsumexp(Vi))
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_noSponsor_M2V3")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) — M2V3 ===")
print(delta.describe())


## Model 2 version 4


In [ ]:
# === Model 2 V4: Demografi × Özellik (Edu×ListenScore, Young×TimeCost, Female×Sponsor) ===
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

# ---------- 0) BigQuery'den M2V4 verisini çek ----------
PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V4"

client = bigquery.Client(project=PROJECT_ID)
query = f"""
SELECT
  user_id,
  podcast_id,
  chosen,
  listen_score_z,
  timecost_z,
  sponsor,
  listen_highEdu,   -- Education×ListenScore (SQL'de numeric üretilmiş)
  time_young,       -- Age(Young)×TimeCost
  sponsor_female,   -- Gender(Female)×Sponsor
  obs_weight
FROM `{TABLE}`
"""
df = client.query(query).to_dataframe()

# ---------- 1) Tipleri sabitle ----------
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)

num_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_highEdu", "time_young", "sponsor_female",
    "obs_weight"
]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# NaN / +/-inf -> 0.0
df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---------- 2) Tasarım matrisi (M1 + M2V4 etkileşimleri) ----------
X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_highEdu", "time_young", "sponsor_female"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "ListenScore×HighEdu", "TimeCost×Young", "Sponsor×Female"
]

# Kullanıcı bazında sıralama ve gruplama
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)

K = X.shape[1]

# ---------- 3) MNL: LL, gradient, hessian ----------
def mnl_obj(beta):
    beta = np.asarray(beta, dtype=np.float64)
    ll = 0.0
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        ll_i = ui[yi==1].sum() - logsumexp(ui)
        ll += w_user[i] * ll_i
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, dtype=np.float64)
    grad = np.zeros(K, dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        x_chosen = Xi[yi==1].sum(axis=0)
        x_bar = Xi.T @ p
        grad += w_user[i] * (x_chosen - x_bar)
    return -grad

def mnl_hessian(beta):
    beta = np.asarray(beta, dtype=np.float64)
    H = np.zeros((K, K), dtype=np.float64)
    for i, (s, c) in enumerate(zip(start_idx, counts)):
        Xi = X[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p = p / p.sum()
        Xw = Xi * p[:, None]
        x_bar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:, None])) - np.outer(x_bar, x_bar)
        H -= w_user[i] * S
    return H

# ---------- 4) MLE ----------
beta0 = np.zeros(K, dtype=np.float64)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# Cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat)
H_inv = np.linalg.inv(H)

meat = np.zeros((K, K), dtype=np.float64)
for i, (s, c) in enumerate(zip(start_idx, counts)):
    Xi = X[s:s+c]; yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p = p / p.sum()
    x_chosen = Xi[yi==1].sum(axis=0)
    x_bar = Xi.T @ p
    score_i = w_user[i] * (x_chosen - x_bar)
    meat += np.outer(score_i, score_i)

cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---------- 5) LL, LL0 (weighted), R^2 ----------
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
mcfadden_r2 = 1 - (LL / LL0)
k = K
N_users = len(counts)
mcfadden_adj = 1 - ((LL - k) / LL0)

# ---------- 6) Sonuç tablosu ----------
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== MNL (Model 2 V4: Demografi × Özellik) — Sonuçlar ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {mcfadden_r2:.4f}")
print(f"McFadden Adj. R^2: {mcfadden_adj:.4f}")

# ---------- 7) CS ve karşı-olgusal (Sponsor=0) ----------
b = dict(zip(param_names, beta_hat))
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
LxEdu = df["listen_highEdu"].to_numpy()
TxYoung = df["time_young"].to_numpy()
SxFem = df["sponsor_female"].to_numpy()

# V: baseline + interactions (M2V4)
V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["ListenScore×HighEdu"]*LxEdu
     + b["TimeCost×Young"]*TxYoung
     + b["Sponsor×Female"]*SxFem)

cs_list = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]
    cs_list.append(logsumexp(Vi))
    ptr += c
cs = pd.Series(cs_list, index=uniq_users, name="CS_M2V4")
print("\n=== Consumer Surplus (per user) — M2V4 ===")
print(cs.describe())

# Counterfactual: Sponsor=0 (Sponsor etkileşimi de 0 olur)
V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + b["ListenScore×HighEdu"]*LxEdu
        + b["TimeCost×Young"]*TxYoung
        + 0.0)

cs_cf_list = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]
    cs_cf_list.append(logsumexp(Vi))
    ptr += c
cs_cf = pd.Series(cs_cf_list, index=uniq_users, name="CS_noSponsor_M2V4")

delta = cs_cf - cs
print("\n=== ΔCS (Sponsor removed) — M2V4 ===")
print(delta.describe())


## Model 2 version 5


In [ ]:
# === M2V5: Edu×TimeCost, Income×ListenScore, Freq×Sponsor ===
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V5"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT user_id,podcast_id,chosen,
       listen_score_z,timecost_z,sponsor,
       time_highEdu,listen_highIncome,sponsor_highFreq,
       obs_weight
FROM `{TABLE}`
"""
df = client.query(q).to_dataframe()

# types
df["chosen"] = df["chosen"].astype(int)
df["sponsor"]= df["sponsor"].astype(int)
num_cols = ["listen_score_z","timecost_z","sponsor","time_highEdu","listen_highIncome","sponsor_highFreq","obs_weight"]
for c in num_cols: df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

# design
order = np.argsort(df["user_id"].to_numpy()); df = df.iloc[order].reset_index(drop=True)
X_cols = ["listen_score_z","timecost_z","sponsor","time_highEdu","listen_highIncome","sponsor_highFreq"]
param_names = ["ListenScore","TimeCost","Sponsor","TimeCost×HighEdu","ListenScore×HighIncome","Sponsor×HighFreq"]

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s,c in zip(start_idx,counts)], dtype=np.float64)
K = X.shape[1]

def mnl_obj(beta):
    beta = np.asarray(beta, np.float64); ll=0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta
        ll += w_user[i]*(ui[yi==1].sum()-logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta=np.asarray(beta,np.float64); g=np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta
        p=np.exp(ui-ui.max()); p/=p.sum()
        g += w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T@p))
    return -g

def mnl_hessian(beta):
    beta=np.asarray(beta,np.float64); H=np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; ui=Xi@beta
        p=np.exp(ui-ui.max()); p/=p.sum()
        Xw=Xi*p[:,None]; xbar=Xw.sum(axis=0)
        S=(Xi.T@(Xi*p[:,None]))-np.outer(xbar,xbar)
        H -= w_user[i]*S
    return H

from numpy.linalg import inv
beta0=np.zeros(K); opt=minimize(mnl_obj,beta0,jac=mnl_score,method="BFGS")
beta_hat=opt.x; H=mnl_hessian(beta_hat); H_inv=inv(H)

meat=np.zeros((K,K))
for i,(s,c) in enumerate(zip(start_idx,counts)):
    Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta_hat
    p=np.exp(ui-ui.max()); p/=p.sum()
    score_i=w_user[i]*((Xi[yi==1].sum(axis=0))-(Xi.T@p))
    meat+=np.outer(score_i,score_i)
cov=H_inv@meat@H_inv; se=np.sqrt(np.diag(cov))

LL=-mnl_obj(beta_hat); LL0=-np.sum(w_user*np.log(counts.astype(float)))
r2=1-(LL/LL0); r2_adj=1-((LL-K)/LL0)

res=pd.DataFrame({"coef":beta_hat,"std_err":se,"z":np.divide(beta_hat,se,out=np.full_like(se,np.nan),where=se>0)},index=param_names)
print("=== M2V5 Results ==="); display(res.round(4))
print(f"LL: {LL:.3f} | LL0(w): {LL0:.3f} | McFadden R^2: {r2:.4f} | Adj: {r2_adj:.4f}")

# CS & Sponsor=0 counterfactual
b=dict(zip(param_names,beta_hat))
L=df["listen_score_z"].to_numpy(); T=df["timecost_z"].to_numpy(); S=df["sponsor"].to_numpy()
TxEdu=df["time_highEdu"].to_numpy(); LxInc=df["listen_highIncome"].to_numpy(); SxFreq=df["sponsor_highFreq"].to_numpy()

V=b["ListenScore"]*L + b["TimeCost"]*T + b["Sponsor"]*S + b["TimeCost×HighEdu"]*TxEdu + b["ListenScore×HighIncome"]*LxInc + b["Sponsor×HighFreq"]*SxFreq
cs=[]; ptr=0
for c in counts:
    Vi=V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr+=c
print(pd.Series(cs,name="CS_M2V5").describe())

V_cf=b["ListenScore"]*L + b["TimeCost"]*T + 0 + b["TimeCost×HighEdu"]*TxEdu + b["ListenScore×HighIncome"]*LxInc + 0
cs_cf=[]; ptr=0
for c in counts:
    Vi=V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr+=c
print(pd.Series(np.array(cs_cf)-np.array(cs),name="ΔCS_noSponsor_M2V5").describe())


## Model 2 version 6


In [ ]:
# === M2V6: Edu×Sponsor, Older×TimeCost ===
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V6"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT user_id,podcast_id,chosen,
       listen_score_z,timecost_z,sponsor,
       sponsor_highEdu,time_older,
       obs_weight
FROM `{TABLE}`
"""
df = client.query(q).to_dataframe()

df["chosen"]=df["chosen"].astype(int)
df["sponsor"]=df["sponsor"].astype(int)
num_cols=["listen_score_z","timecost_z","sponsor","sponsor_highEdu","time_older","obs_weight"]
for c in num_cols: df[c]=pd.to_numeric(df[c],errors="coerce")
df[num_cols]=df[num_cols].replace([np.inf,-np.inf],np.nan).fillna(0.0)

order=np.argsort(df["user_id"].to_numpy()); df=df.iloc[order].reset_index(drop=True)

X_cols=["listen_score_z","timecost_z","sponsor","sponsor_highEdu","time_older"]
param_names=["ListenScore","TimeCost","Sponsor","Sponsor×HighEdu","TimeCost×Older"]

X=df[X_cols].to_numpy(dtype=np.float64)
y=df["chosen"].to_numpy(dtype=int)
users=df["user_id"].to_numpy()
w_row=df["obs_weight"].to_numpy(dtype=float)

uniq_users,start_idx,counts=np.unique(users,return_index=True,return_counts=True)
w_user=np.array([w_row[s:s+c].mean() for s,c in zip(start_idx,counts)],dtype=np.float64)
K=X.shape[1]

def mnl_obj(b):
    b=np.asarray(b,np.float64); ll=0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@b
        ll+=w_user[i]*(ui[yi==1].sum()-logsumexp(ui))
    return -ll

def mnl_score(b):
    b=np.asarray(b,np.float64); g=np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@b
        p=np.exp(ui-ui.max()); p/=p.sum()
        g+=w_user[i]*((Xi[yi==1].sum(axis=0))-(Xi.T@p))
    return -g

def mnl_hessian(b):
    b=np.asarray(b,np.float64); H=np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; ui=Xi@b
        p=np.exp(ui-ui.max()); p/=p.sum()
        Xw=Xi*p[:,None]; xbar=Xw.sum(axis=0)
        S=(Xi.T@(Xi*p[:,None]))-np.outer(xbar,xbar)
        H-=w_user[i]*S
    return H

from numpy.linalg import inv
beta0=np.zeros(K); opt=minimize(mnl_obj,beta0,jac=mnl_score,method="BFGS")
beta_hat=opt.x; H=mnl_hessian(beta_hat); H_inv=inv(H)

meat=np.zeros((K,K))
for i,(s,c) in enumerate(zip(start_idx,counts)):
    Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta_hat
    p=np.exp(ui-ui.max()); p/=p.sum()
    score_i=w_user[i]*((Xi[yi==1].sum(axis=0))-(Xi.T@p))
    meat+=np.outer(score_i,score_i)
cov=H_inv@meat@H_inv; se=np.sqrt(np.diag(cov))

LL=-mnl_obj(beta_hat); LL0=-np.sum(w_user*np.log(counts.astype(float)))
r2=1-(LL/LL0); r2_adj=1-((LL-K)/LL0)

res=pd.DataFrame({"coef":beta_hat,"std_err":se,"z":np.divide(beta_hat,se,out=np.full_like(se,np.nan),where=se>0)},index=param_names)
print("=== M2V6 Results ==="); display(res.round(4))
print(f"LL: {LL:.3f} | LL0(w): {LL0:.3f} | McFadden R^2: {r2:.4f} | Adj: {r2_adj:.4f}")

# CS & Sponsor=0 counterfactual
b=dict(zip(param_names,beta_hat))
L=df["listen_score_z"].to_numpy(); T=df["timecost_z"].to_numpy(); S=df["sponsor"].to_numpy()
SxEdu=df["sponsor_highEdu"].to_numpy(); TxOld=df["time_older"].to_numpy()

V=b["ListenScore"]*L + b["TimeCost"]*T + b["Sponsor"]*S + b["Sponsor×HighEdu"]*SxEdu + b["TimeCost×Older"]*TxOld
cs=[]; ptr=0
for c in counts:
    Vi=V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr+=c
print(pd.Series(cs,name="CS_M2V6").describe())

V_cf=b["ListenScore"]*L + b["TimeCost"]*T + 0 + 0 + b["TimeCost×Older"]*TxOld
cs_cf=[]; ptr=0
for c in counts:
    Vi=V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr+=c
print(pd.Series(np.array(cs_cf)-np.array(cs),name="ΔCS_noSponsor_M2V6").describe())


## Model 2 version 7


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from numpy.linalg import inv

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V7"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT user_id, podcast_id, chosen,
       listen_score_z, timecost_z, sponsor,
       listen_older, listen_female,
       obs_weight
FROM `{TABLE}`
"""
df = client.query(q).to_dataframe()

# ---- tipleri sabitle ----
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)
num_cols = ["listen_score_z","timecost_z","sponsor","listen_older","listen_female","obs_weight"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---- tasarım matrisi ----
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "listen_older", "listen_female"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "ListenScore×Older", "ListenScore×Female"
]

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)
K = X.shape[1]

# ---- MNL: LL, gradient, hessian ----
def mnl_obj(beta):
    beta = np.asarray(beta, np.float64); ll = 0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        ll += w_user[i]*(ui[yi==1].sum() - logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, np.float64); g = np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        g += w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T @ p))
    return -g

def mnl_hessian(beta):
    beta = np.asarray(beta, np.float64); H = np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        Xw = Xi * p[:,None]; xbar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:,None])) - np.outer(xbar, xbar)
        H -= w_user[i]*S
    return H

# ---- MLE ----
beta0 = np.zeros(K)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat); H_inv = inv(H)
meat = np.zeros((K,K))
for i,(s,c) in enumerate(zip(start_idx,counts)):
    Xi = X[s:s+c]; yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p /= p.sum()
    score_i = w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T @ p))
    meat += np.outer(score_i, score_i)
cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---- LL, LL0 (weighted), R^2 ----
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
r2 = 1 - (LL / LL0)
r2_adj = 1 - ((LL - K) / LL0)

# ---- sonuç tablosu ----
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== M2V7 Results ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {r2:.4f} | Adjusted: {r2_adj:.4f}")

# ---- CS ve (opsiyonel) sponsor=0 karşı-olgusal (M2V7'de sponsor interaction yok ama kıyas için bırakıyorum) ----
b = dict(zip(param_names, beta_hat))
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
LxOld = df["listen_older"].to_numpy()
LxFem = df["listen_female"].to_numpy()

V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["ListenScore×Older"]*LxOld
     + b["ListenScore×Female"]*LxFem)

cs = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr += c
print(pd.Series(cs, name="CS_M2V7").describe())

# counterfactual: Sponsor=0 (interaction yok → sadece sponsor terimi kapanır)
V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + b["ListenScore×Older"]*LxOld
        + b["ListenScore×Female"]*LxFem)

cs_cf = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr += c
print(pd.Series(np.array(cs_cf)-np.array(cs), name="ΔCS_noSponsor_M2V7").describe())


## Model 2 version 8


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from numpy.linalg import inv

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V8"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT user_id, podcast_id, chosen,
       listen_score_z, timecost_z, sponsor,
       sponsor_lowIncome, sponsor_highIncome,
       obs_weight
FROM `{TABLE}`
"""
df = client.query(q).to_dataframe()

# ---- tipleri sabitle ----
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)
num_cols = ["listen_score_z","timecost_z","sponsor","sponsor_lowIncome","sponsor_highIncome","obs_weight"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---- tasarım matrisi ----
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X_cols = [
    "listen_score_z", "timecost_z", "sponsor",
    "sponsor_lowIncome", "sponsor_highIncome"
]
param_names = [
    "ListenScore", "TimeCost", "Sponsor",
    "Sponsor×LowIncome", "Sponsor×HighIncome"
]

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)
K = X.shape[1]

# ---- MNL: LL, gradient, hessian ----
def mnl_obj(beta):
    beta = np.asarray(beta, np.float64); ll = 0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        ll += w_user[i]*(ui[yi==1].sum() - logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, np.float64); g = np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]
        ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        g += w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T @ p))
    return -g

def mnl_hessian(beta):
    beta = np.asarray(beta, np.float64); H = np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        Xw = Xi * p[:,None]; xbar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:,None])) - np.outer(xbar, xbar)
        H -= w_user[i]*S
    return H

# ---- MLE ----
beta0 = np.zeros(K)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# cluster-robust (user) SE: sandwich
H = mnl_hessian(beta_hat); H_inv = inv(H)
meat = np.zeros((K,K))
for i,(s,c) in enumerate(zip(start_idx,counts)):
    Xi = X[s:s+c]; yi = y[s:s+c]
    ui = Xi @ beta_hat
    p = np.exp(ui - ui.max()); p /= p.sum()
    score_i = w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T @ p))
    meat += np.outer(score_i, score_i)
cov = H_inv @ meat @ H_inv
se = np.sqrt(np.diag(cov))

# ---- LL, LL0 (weighted), R^2 ----
LL  = -mnl_obj(beta_hat)
LL0 = -np.sum(w_user * np.log(counts.astype(float)))
r2 = 1 - (LL / LL0)
r2_adj = 1 - ((LL - K) / LL0)

# ---- sonuç tablosu ----
res = pd.DataFrame({
    "coef": beta_hat,
    "std_err": se,
    "z": np.divide(beta_hat, se, out=np.full_like(se, np.nan), where=se>0)
}, index=param_names)

print("=== M2V8 Results — Sponsor × Income (Mid ref) ===")
display(res.round(4))
print(f"Log-Likelihood: {LL:.3f}")
print(f"Null Log-Likelihood (weighted): {LL0:.3f}")
print(f"McFadden R^2: {r2:.4f} | Adjusted: {r2_adj:.4f}")

# ---- CS ve Sponsor=0 karşı-olgusal ----
b = dict(zip(param_names, beta_hat))
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
SxLow  = df["sponsor_lowIncome"].to_numpy()
SxHigh = df["sponsor_highIncome"].to_numpy()

V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor"]*S
     + b["Sponsor×LowIncome"]*SxLow
     + b["Sponsor×HighIncome"]*SxHigh)

cs = []; ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr += c
print(pd.Series(cs, name="CS_M2V8").describe())

V_cf = (b["ListenScore"]*L
        + b["TimeCost"]*T
        + 0.0
        + 0.0
        + 0.0)
cs_cf = []; ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr += c
print(pd.Series(np.array(cs_cf)-np.array(cs), name="ΔCS_noSponsor_M2V8").describe())


## Model 2 version 9


In [ ]:
# === M2V10: Sponsor × Gender (Male ref) ===
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from numpy.linalg import inv

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V10"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT user_id, podcast_id, chosen,
       listen_score_z, timecost_z, sponsor,
       sponsor_female,
       obs_weight
FROM `{TABLE}`
"""
df = client.query(q).to_dataframe()

# types
df["chosen"]=df["chosen"].astype(int)
df["sponsor"]=df["sponsor"].astype(int)
num_cols=["listen_score_z","timecost_z","sponsor","sponsor_female","obs_weight"]
for c in num_cols: df[c]=pd.to_numeric(df[c],errors="coerce")
df[num_cols]=df[num_cols].replace([np.inf,-np.inf],np.nan).fillna(0.0)

print(df.shape,"users:",df["user_id"].nunique(),"alts:",df["podcast_id"].nunique())
display(df.head(3))

# design
order=np.argsort(df["user_id"].to_numpy())
df=df.iloc[order].reset_index(drop=True)

X_cols=["listen_score_z","timecost_z","sponsor","sponsor_female"]
param_names=["ListenScore","TimeCost","Sponsor(Male ref)","Sponsor×Female"]

X=df[X_cols].to_numpy(dtype=np.float64)
y=df["chosen"].to_numpy(dtype=int)
users=df["user_id"].to_numpy()
w_row=df["obs_weight"].to_numpy(dtype=float)

uniq_users,start_idx,counts=np.unique(users,return_index=True,return_counts=True)
w_user=np.array([w_row[s:s+c].mean() for s,c in zip(start_idx,counts)],dtype=np.float64)
K=X.shape[1]

# loglikelihood, gradient, hessian
def mnl_obj(beta):
    beta=np.asarray(beta,np.float64); ll=0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta
        ll+=w_user[i]*(ui[yi==1].sum()-logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta=np.asarray(beta,np.float64); g=np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta
        p=np.exp(ui-ui.max()); p/=p.sum()
        g+=w_user[i]*((Xi[yi==1].sum(axis=0))-(Xi.T@p))
    return -g

def mnl_hessian(beta):
    beta=np.asarray(beta,np.float64); H=np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi=X[s:s+c]; ui=Xi@beta
        p=np.exp(ui-ui.max()); p/=p.sum()
        Xw=Xi*p[:,None]; xbar=Xw.sum(axis=0)
        S=(Xi.T@(Xi*p[:,None]))-np.outer(xbar,xbar)
        H-=w_user[i]*S
    return H

beta0=np.zeros(K); opt=minimize(mnl_obj,beta0,jac=mnl_score,method="BFGS")
beta_hat=opt.x; H=mnl_hessian(beta_hat); H_inv=inv(H)

# sandwich SE
meat=np.zeros((K,K))
for i,(s,c) in enumerate(zip(start_idx,counts)):
    Xi=X[s:s+c]; yi=y[s:s+c]; ui=Xi@beta_hat
    p=np.exp(ui-ui.max()); p/=p.sum()
    score_i=w_user[i]*((Xi[yi==1].sum(axis=0))-(Xi.T@p))
    meat+=np.outer(score_i,score_i)
cov=H_inv@meat@H_inv; se=np.sqrt(np.diag(cov))

LL=-mnl_obj(beta_hat); LL0=-np.sum(w_user*np.log(counts.astype(float)))
r2=1-(LL/LL0); r2_adj=1-((LL-K)/LL0)

res=pd.DataFrame({"coef":beta_hat,"std_err":se,"z":np.divide(beta_hat,se,out=np.full_like(se,np.nan),where=se>0)},index=param_names)
print("=== M2V10 Results — Sponsor × Gender (Male ref) ===")
display(res.round(4))
print(f"LL: {LL:.3f} | LL0(w): {LL0:.3f} | McFadden R^2: {r2:.4f} | Adj: {r2_adj:.4f}")

# consumer surplus
b=dict(zip(param_names,beta_hat))
L=df["listen_score_z"].to_numpy(); T=df["timecost_z"].to_numpy(); S=df["sponsor"].to_numpy()
SxFem=df["sponsor_female"].to_numpy()

V=(b["ListenScore"]*L + b["TimeCost"]*T + b["Sponsor(Male ref)"]*S + b["Sponsor×Female"]*SxFem)
cs=[]; ptr=0
for c in counts:
    Vi=V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr+=c
print(pd.Series(cs,name="CS_M2V10").describe())

V_cf=(b["ListenScore"]*L + b["TimeCost"]*T + 0.0)
cs_cf=[]; ptr=0
for c in counts:
    Vi=V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr+=c
print(pd.Series(np.array(cs_cf)-np.array(cs),name="ΔCS_noSponsor_M2V10").describe())


## Model 2 version 10


In [ ]:
!pip -q install google-cloud-bigquery google-auth scipy

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd, numpy as np
from scipy.optimize import minimize
from scipy.special import logsumexp
from numpy.linalg import inv
from scipy import stats

PROJECT_ID = "my-dissertation-470916"
TABLE = "my-dissertation-470916.podcast_primary_bucket.panel_match_only_est_M2V10"
USERS = "my-dissertation-470916.podcast_primary_bucket.users_clean"

client = bigquery.Client(project=PROJECT_ID)
q = f"""
SELECT
  p.user_id, p.podcast_id, p.chosen,
  p.listen_score_z, p.timecost_z, p.sponsor,
  p.sponsor_female, p.obs_weight,
  -- cinsiyet bayrağı (0=Male, 1=Female)
  CAST(IF(LOWER(TRIM(u.gender))='female', 1, 0) AS INT64) AS female_flag
FROM `{TABLE}` p
LEFT JOIN `{USERS}` u
USING (user_id)
"""
df = client.query(q).to_dataframe()

# Tipleri sabitle
df["chosen"]  = df["chosen"].astype(int)
df["sponsor"] = df["sponsor"].astype(int)
num_cols = ["listen_score_z","timecost_z","sponsor","sponsor_female","obs_weight","female_flag"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

print(df.shape, "users:", df["user_id"].nunique(), "alts:", df["podcast_id"].nunique())
display(df.head(3))

# ---- M2V10 tasarım matrisi ----
order = np.argsort(df["user_id"].to_numpy())
df = df.iloc[order].reset_index(drop=True)

X_cols = ["listen_score_z","timecost_z","sponsor","sponsor_female"]
param_names = ["ListenScore","TimeCost","Sponsor(Male ref)","Sponsor×Female"]

X = df[X_cols].to_numpy(dtype=np.float64)
y = df["chosen"].to_numpy(dtype=int)
users = df["user_id"].to_numpy()
w_row = df["obs_weight"].to_numpy(dtype=float)

uniq_users, start_idx, counts = np.unique(users, return_index=True, return_counts=True)
w_user = np.array([w_row[s:s+c].mean() for s, c in zip(start_idx, counts)], dtype=np.float64)
K = X.shape[1]

# ---- MNL: LL, gradient, hessian ----
def mnl_obj(beta):
    beta = np.asarray(beta, np.float64); ll = 0.0
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]; ui = Xi @ beta
        ll += w_user[i]*(ui[yi==1].sum() - logsumexp(ui))
    return -ll

def mnl_score(beta):
    beta = np.asarray(beta, np.float64); g = np.zeros(K)
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; yi = y[s:s+c]; ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        g += w_user[i]*((Xi[yi==1].sum(axis=0)) - (Xi.T @ p))
    return -g

def mnl_hessian(beta):
    beta = np.asarray(beta, np.float64); H = np.zeros((K,K))
    for i,(s,c) in enumerate(zip(start_idx,counts)):
        Xi = X[s:s+c]; ui = Xi @ beta
        p = np.exp(ui - ui.max()); p /= p.sum()
        Xw = Xi * p[:,None]; xbar = Xw.sum(axis=0)
        S = (Xi.T @ (Xi * p[:,None])) - np.outer(xbar, xbar)
        H -= w_user[i]*S
    return H

# ---- Tahmin ----
beta0 = np.zeros(K)
opt = minimize(mnl_obj, beta0, jac=mnl_score, method="BFGS")
beta_hat = opt.x

# ---- CS ve Sponsor=0 karşı-olgusal ----
b = dict(zip(param_names, beta_hat))
L = df["listen_score_z"].to_numpy()
T = df["timecost_z"].to_numpy()
S = df["sponsor"].to_numpy()
SxFem = df["sponsor_female"].to_numpy()

V = (b["ListenScore"]*L
     + b["TimeCost"]*T
     + b["Sponsor(Male ref)"]*S
     + b["Sponsor×Female"]*SxFem)

cs = []
ptr = 0
for c in counts:
    Vi = V[ptr:ptr+c]; cs.append(logsumexp(Vi)); ptr += c
cs = pd.Series(cs, index=uniq_users, name="CS_M2V10")

V_cf = (b["ListenScore"]*L + b["TimeCost"]*T + 0.0)  # sponsor=0, interaction=0
cs_cf = []
ptr = 0
for c in counts:
    Vi = V_cf[ptr:ptr+c]; cs_cf.append(logsumexp(Vi)); ptr += c
cs_cf = pd.Series(cs_cf, index=uniq_users, name="CS_noSponsor_M2V10")

delta = (cs_cf - cs).rename("ΔCS")

# ---- cinsiyeti user-level'e düşür ----
female_per_user = df.groupby("user_id")["female_flag"].max().rename("female_flag")
out = pd.concat([delta, cs, cs_cf, female_per_user], axis=1)

print("\n=== ΔCS by Gender ===")
print(out.groupby("female_flag")["ΔCS"].describe())

# Basit fark testi (Welch t-test)
delta_male   = out.loc[out["female_flag"]==0, "ΔCS"].to_numpy()
delta_female = out.loc[out["female_flag"]==1, "ΔCS"].to_numpy()
tstat, pval = stats.ttest_ind(delta_male, delta_female, equal_var=False, nan_policy='omit')
print(f"\nWelch t-test (ΔCS_male vs ΔCS_female): t = {tstat:.3f}, p = {pval:.4f}")

# Opsiyonel: yüzde fark
mean_male, mean_fem = np.nanmean(delta_male), np.nanmean(delta_female)
print(f"\nMean ΔCS (male):   {mean_male:.4f}")
print(f"Mean ΔCS (female): {mean_fem:.4f}")
print(f"Female vs Male ΔCS difference: {mean_fem - mean_male:+.4f}")


## Sponsor affinity diagnostic


In [ ]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="my-dissertation-470916")
aff = client.query("""
SELECT * FROM `my-dissertation-470916.podcast_primary_bucket.user_sponsor_affinity`
""").to_dataframe()

print("=== Sponsor affinity (lift) summary ===")
print(aff["sponsor_lift"].describe())
print("\nShare of users with lift > 0:", (aff["sponsor_lift"]>0).mean())
print("Share of users with lift < 0:", (aff["sponsor_lift"]<0).mean())
